In [2]:

from google.colab import drive
drive.mount('/content/drive')

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

import torchvision
import torchvision.transforms as T
from torchvision.models.detection.ssd import SSDClassificationHead
from torchvision.models.detection import _utils
from torchvision.models.detection import SSD300_VGG16_Weights
from torchvision.models.detection import ssd300_vgg16
from torchvision.ops import nms

from sklearn.metrics import average_precision_score

import pandas as pd
from PIL import Image
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time
import os


os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

Mounted at /content/drive


In [3]:
class CustomDataset(Dataset):
    def __init__(self, video_path, annotations_path, transform=None, start_frame=0, end_frame=None):
        self.video_path = video_path
        self.annotations = pd.read_csv(annotations_path, sep='\s+', header=None,
                                       names=['track_id', 'xmin', 'ymin', 'xmax', 'ymax',
                                              'frame', 'lost', 'occluded', 'generated', 'label'])
        self.transform = transform
        self.cap = cv2.VideoCapture(video_path)
        self.start_frame = start_frame
        if end_frame is not None:
            self.end_frame = end_frame
        else:
            self.end_frame = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))

    def __len__(self):
        return self.end_frame - self.start_frame

    def __getitem__(self, idx):
        # Adjust index based on starting frame and open to start index
        actual_idx = idx + self.start_frame
        self.cap.set(cv2.CAP_PROP_POS_FRAMES, actual_idx)
        success, frame = self.cap.read()

        image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        original_size = image.size  # Store the original size (width, height)

        if self.transform:
            image = self.transform(image)

        current_annotations = self.annotations[self.annotations['frame'] == actual_idx + 1]

        # Prepare target
        if not current_annotations.empty:
            visible_annotations = current_annotations[current_annotations['lost'] == 0]
            boxes = visible_annotations[['xmin', 'ymin', 'xmax', 'ymax']].values.astype(np.float32)
            labels = visible_annotations['label'].str.strip('"').values

            # Transform bounding boxes to match the resized image
            if self.transform:
                width_ratio = image.shape[1] / original_size[0]
                height_ratio = image.shape[2] / original_size[1]
                boxes[:, [0, 2]] *= width_ratio  # Scale xmin and xmax
                boxes[:, [1, 3]] *= height_ratio  # Scale ymin and ymax
                # print('width_ratio: ',width_ratio)
                # print('height_ratio: ',height_ratio)
                # print(original_size)
                # print('width_ratio ', image.shape[1], original_size[0])
                # print('height_ratio ', image.shape[2], original_size[1])

            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor([self.get_label_mapping(label) for label in labels], dtype=torch.int64)
        else:
            boxes = torch.empty((0, 4), dtype=torch.float32)
            labels = torch.empty((0,), dtype=torch.int64)

        target = {'boxes': boxes, 'labels': labels}

        # print('width_ratio: ',width_ratio)
        # print('height_ratio: ',height_ratio)

        return image, target

    def get_label_mapping(self, label):
        label_map = {"Pedestrian": 1, "Car": 2, "Biker": 3, "Bus": 4, "Cart": 5, "Skater": 6}
        return label_map.get(label, 0)  # Default to 0 if not found



<>:4: SyntaxWarning: invalid escape sequence '\s'
<>:4: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipython-input-3272934415.py:4: SyntaxWarning: invalid escape sequence '\s'
  self.annotations = pd.read_csv(annotations_path, sep='\s+', header=None,


In [4]:
class CombinedDataset(Dataset):
    def __init__(self, datasets):
        self.datasets = datasets

    def __len__(self):
        return sum(len(dataset) for dataset in self.datasets)

    def __getitem__(self, idx):
        cumulative_length = 0

        for dataset in self.datasets:
            dataset_length = len(dataset)
            if idx < cumulative_length + dataset_length:
                return dataset[idx - cumulative_length]
            cumulative_length += dataset_length

        raise IndexError("Index out of bounds.")

In [5]:

# Training function
def train_model(model, dataset, num_epochs=10, learning_rate=0.001, validation_dataset=None, momentum_in=0.9, decay=5e-4):
    # Load data
    data_loader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=my_collate_fn)
    # Define optimizer
    optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=momentum_in, weight_decay=decay)

    training_time = 0
    for epoch in range(num_epochs):
        time1 = time.time()
        model.train()  # Set the model to training mode
        total_loss = 0

        for images, targets in data_loader:
            # Store images and targets as lists
            images = [image.to(device) for image in images]
            targets = [{k: v.to(device) for k, v in target.items()} for target in targets]

            # Zero parameter gradients
            optimizer.zero_grad()

            # Forward pass
            loss_dict = model(images, targets)
            # print(loss_dict)
            losses = sum(loss for loss in loss_dict.values())

            # Backward pass
            losses.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)

            optimizer.step()

            total_loss += losses.item()
            current_loss = losses.item()

            # print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {current_loss:.4f}')
        time2 = time.time()
        time_elapsed = time2 - time1
        training_time += time_elapsed
        print(f'Epoch [{epoch + 1}/{num_epochs}], Avg Epoch Loss: {total_loss / len(data_loader)}, Time: {time_elapsed} seconds')

        if validation_dataset is not None:
            validate_model(model, validation_dataset)
    print('Total Train Time: ', training_time)

# Define the validation function
def validate_model(model, dataset):
    data_loader = DataLoader(dataset, batch_size=32, shuffle=False, collate_fn=my_collate_fn)

    # Set the model to train mode to obtain losses
    model.train()
    total_loss = 0
    num_batches = 0

    with torch.no_grad():  # Disable gradient calculation for validation
        for images, targets in data_loader:

            images = [image.to(device) for image in images]
            targets = [{k: v.to(device) for k, v in target.items()} for target in targets]

            # Forward pass
            loss_dict = model(images, targets)
            # print(loss_dict)
            losses = sum(loss for loss in loss_dict.values())

            # Accumulate total loss
            total_loss += losses.item()
            num_batches += 1

    # Calculate and print average loss
    avg_loss = total_loss / num_batches if num_batches > 0 else float('inf')
    print(f'Validation Avg Loss: {avg_loss:.4f}')

def my_collate_fn(batch):
    images, targets = zip(*batch)  # Unzip the batch
    return list(images), list(targets)  # Return as lists



In [6]:

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Open the video file and the annotations file

# Train and Valid datasets
video_path1 = r"/content/drive/MyDrive/stanford_campus_dataset/videos/bookstore/video0/video.mov"
annotations_path1 = r"/content/drive/MyDrive/stanford_campus_dataset/annotations/bookstore/video0/annotations.txt"
video_path2 = r"/content/drive/MyDrive/stanford_campus_dataset/videos/deathCircle/video0/video.mov"
annotations_path2 = r"/content/drive/MyDrive/stanford_campus_dataset/annotations/deathCircle/video0/annotations.txt"
video_path3 = r"/content/drive/MyDrive/stanford_campus_dataset/videos/coupa/video0/video.mov"
annotations_path3 = r"/content/drive/MyDrive/stanford_campus_dataset/annotations/coupa/video0/annotations.txt"
video_path4 = r"/content/drive/MyDrive/stanford_campus_dataset/videos/little/video0/video.mov"
annotations_path4 = r"/content/drive/MyDrive/stanford_campus_dataset/annotations/little/video0/annotations.txt"
video_path5 = r"/content/drive/MyDrive/stanford_campus_dataset/videos/gates/video0/video.mov"
annotations_path5 = r"/content/drive/MyDrive/stanford_campus_dataset/annotations/gates/video0/annotations.txt"
video_path6 = r"/content/drive/MyDrive/stanford_campus_dataset/videos/hyang/video0/video.mov"
annotations_path6 = r"/content/drive/MyDrive/stanford_campus_dataset/annotations/hyang/video0/annotations.txt"
video_path7 = r"/content/drive/MyDrive/stanford_campus_dataset/videos/nexus/video0/video.mov"
annotations_path7 = r"/content/drive/MyDrive/stanford_campus_dataset/annotations/nexus/video0/annotations.txt"
video_path8 = r"/content/drive/MyDrive/stanford_campus_dataset/videos/quad/video0/video.mov"
annotations_path8 = r"/content/drive/MyDrive/stanford_campus_dataset/annotations/quad/video0/annotations.txt"

# Test dataset
video_path10 = r"/content/drive/MyDrive/stanford_campus_dataset/videos/bookstore/video1/video.mov"
annotations_path10 = r"/content/drive/MyDrive/stanford_campus_dataset/annotations/bookstore/video1/annotations.txt"

cap1 = cv2.VideoCapture(video_path1)
cap2 = cv2.VideoCapture(video_path2)
cap3 = cv2.VideoCapture(video_path2)
cap4 = cv2.VideoCapture(video_path2)
cap5 = cv2.VideoCapture(video_path2)
cap6 = cv2.VideoCapture(video_path2)
cap7 = cv2.VideoCapture(video_path2)
cap8 = cv2.VideoCapture(video_path2)
cap10 = cv2.VideoCapture(video_path2)

# Instantiate the dataset
total_frames1 = int(cap1.get(cv2.CAP_PROP_FRAME_COUNT))
print(total_frames1)
total_frames2 = int(cap2.get(cv2.CAP_PROP_FRAME_COUNT))
print(total_frames2)
total_frames3 = int(cap1.get(cv2.CAP_PROP_FRAME_COUNT))
print(total_frames3)
total_frames4 = int(cap2.get(cv2.CAP_PROP_FRAME_COUNT))
print(total_frames4)
total_frames5 = int(cap1.get(cv2.CAP_PROP_FRAME_COUNT))
print(total_frames5)
total_frames6 = int(cap2.get(cv2.CAP_PROP_FRAME_COUNT))
print(total_frames6)
total_frames7 = int(cap1.get(cv2.CAP_PROP_FRAME_COUNT))
print(total_frames7)
total_frames8 = int(cap2.get(cv2.CAP_PROP_FRAME_COUNT))
print(total_frames8)
total_frames10 = int(cap1.get(cv2.CAP_PROP_FRAME_COUNT))
print(total_frames10)
# train_end = int(total_frames1 * 0.1)  # First X% for testing
# print(train_end)
# valid_start = train_end
# valid_end = int(total_frames1 * 0.2)  # Next Y% for validation
# print(valid_end-train_end)
pct_video = 0.01
train_end1 = int(total_frames1 * pct_video)
train_end2 = int(total_frames2 * pct_video)
train_end3 = int(total_frames3 * pct_video)
train_end4 = int(total_frames4 * pct_video)
train_end5 = int(total_frames5 * pct_video)
train_end6 = int(total_frames6 * pct_video)
train_end7 = int(total_frames7 * pct_video)
train_end8 = int(total_frames8 * pct_video)
print(train_end1 + train_end2 + train_end3 + train_end4 + train_end5 + train_end6 + train_end7 + train_end8)

valid_end1 = int(total_frames1 * 2 * pct_video)
valid_end2 = int(total_frames2 * 2 * pct_video)
valid_end3 = int(total_frames3 * 2 * pct_video)
valid_end4 = int(total_frames4 * 2 * pct_video)
valid_end5 = int(total_frames5 * 2 * pct_video)
valid_end6 = int(total_frames6 * 2 * pct_video)
valid_end7 = int(total_frames7 * 2 * pct_video)
valid_end8 = int(total_frames8 * 2 * pct_video)
print(valid_end1 + valid_end2 + valid_end3 + valid_end4 + valid_end5 + valid_end6 + valid_end7 + valid_end8
      - (train_end1 + train_end2 + train_end3 + train_end4 + train_end5 + train_end6 + train_end7 + train_end8))

# Transform for training data to be 256x256 for input to
resize_size = 512
transform = T.Compose([
    T.Resize((resize_size, resize_size)),
    T.ToTensor(),
])


# train_dataset = CustomDataset(video_path, annotations_path, transform=transform, end_frame=train_end)
# test_dataset = CustomDataset(video_path, annotations_path, transform=transform, start_frame=test_start, end_frame=test_end)

train_dataset1 = CustomDataset(video_path1, annotations_path1, transform=transform, end_frame=train_end1)
train_dataset2 = CustomDataset(video_path2, annotations_path2, transform=transform, end_frame=train_end2)
train_dataset3 = CustomDataset(video_path3, annotations_path3, transform=transform, end_frame=train_end3)
train_dataset4 = CustomDataset(video_path4, annotations_path4, transform=transform, end_frame=train_end4)
train_dataset5 = CustomDataset(video_path5, annotations_path5, transform=transform, end_frame=train_end5)
train_dataset6 = CustomDataset(video_path6, annotations_path6, transform=transform, end_frame=train_end6)
train_dataset7 = CustomDataset(video_path7, annotations_path7, transform=transform, end_frame=train_end7)
train_dataset8 = CustomDataset(video_path8, annotations_path8, transform=transform, end_frame=train_end8)
valid_dataset1 = CustomDataset(video_path1, annotations_path1, transform=transform, start_frame=train_end1, end_frame=valid_end1)
valid_dataset2 = CustomDataset(video_path2, annotations_path2, transform=transform, start_frame=train_end2, end_frame=valid_end2)
valid_dataset3 = CustomDataset(video_path3, annotations_path3, transform=transform, start_frame=train_end3, end_frame=valid_end3)
valid_dataset4 = CustomDataset(video_path4, annotations_path4, transform=transform, start_frame=train_end4, end_frame=valid_end4)
valid_dataset5 = CustomDataset(video_path5, annotations_path5, transform=transform, start_frame=train_end5, end_frame=valid_end5)
valid_dataset6 = CustomDataset(video_path6, annotations_path6, transform=transform, start_frame=train_end6, end_frame=valid_end6)
valid_dataset7 = CustomDataset(video_path7, annotations_path7, transform=transform, start_frame=train_end7, end_frame=valid_end7)
valid_dataset8 = CustomDataset(video_path8, annotations_path8, transform=transform, start_frame=train_end8, end_frame=valid_end8)

train_dataset = CombinedDataset([train_dataset1, train_dataset2, train_dataset3, train_dataset4, train_dataset5, train_dataset6, train_dataset7, train_dataset8])
valid_dataset = CombinedDataset([valid_dataset1, valid_dataset2, valid_dataset3, valid_dataset4, valid_dataset5, valid_dataset6, valid_dataset7, valid_dataset8])
# train_dataset = CustomDataset(video_path1, annotations_path1, transform=transform)
test_dataset = CustomDataset(video_path10, annotations_path10, transform=transform)

13335
12721
13335
12721
13335
12721
13335
12721
13335
1040
1040


In [7]:


# https://debuggercafe.com/train-ssd300-vgg16/

size = resize_size
num_classes = 7  # classification labels 1-6 + background class label 0

# Load pretrained model
model = torchvision.models.detection.ssd300_vgg16(weights=SSD300_VGG16_Weights.COCO_V1)

# Define classification head
in_channels = _utils.retrieve_out_channels(model.backbone, (size, size))
num_anchors = model.anchor_generator.num_anchors_per_location()
model.head.classification_head = SSDClassificationHead(in_channels=in_channels,
                                                       num_anchors=num_anchors,
                                                       num_classes=num_classes)

# Transform for images
model.transform.min_size = (size,)
model.transform.max_size = size

# Move model to device
model = model.to(device)

# Train model
print('Training begins.')
start_time = time.time()
train_model(model, train_dataset, num_epochs=20, learning_rate=0.1, validation_dataset=valid_dataset)
end_time = time.time()
elapsed_time = end_time - start_time
print(f'Training & Validation time: {elapsed_time:.2f} seconds')
print('Training complete.')

model_path = r"/content/drive/MyDrive/SSD_Object_Detection/ssd300_vgg16_trained_weights_lr1_stratsmall.pth"
torch.save(model.state_dict(), model_path)  # Save the model's state dict

print(f'Model weights saved to {model_path}.')


Training begins.
Epoch [1/20], Avg Epoch Loss: 9.577443946491588, Time: 162.61916613578796 seconds
Validation Avg Loss: 6.9456
Epoch [2/20], Avg Epoch Loss: 6.579504027511135, Time: 162.52784776687622 seconds
Validation Avg Loss: 6.3240
Epoch [3/20], Avg Epoch Loss: 5.802713119622433, Time: 162.5513095855713 seconds
Validation Avg Loss: 5.5453
Epoch [4/20], Avg Epoch Loss: 5.227760994073116, Time: 163.06271529197693 seconds
Validation Avg Loss: 5.4542
Epoch [5/20], Avg Epoch Loss: 4.948980215824012, Time: 160.78190302848816 seconds
Validation Avg Loss: 5.4797
Epoch [6/20], Avg Epoch Loss: 4.495101191780784, Time: 162.6600844860077 seconds
Validation Avg Loss: 4.9801
Epoch [7/20], Avg Epoch Loss: 3.708150141166918, Time: 161.39841270446777 seconds
Validation Avg Loss: 4.5744
Epoch [8/20], Avg Epoch Loss: 2.957210071159132, Time: 162.59954047203064 seconds
Validation Avg Loss: 4.4660
Epoch [9/20], Avg Epoch Loss: 2.252532550782868, Time: 162.0578374862671 seconds
Validation Avg Loss: 5.4

In [ ]:
# model_path = r"/content/drive/MyDrive/SSD_Object_Detection/ssd300_vgg16_trained_weights.pth"
# torch.save(model.state_dict(), model_path)  # Save the model's state dict

# print(f'Model weights saved to {model_path}.')

Model weights saved to /content/drive/MyDrive/SSD_Object_Detection/ssd300_vgg16_trained_weights.pth.


In [7]:
# Load saved model with trained weights
model_path = r"/content/drive/MyDrive/SSD_Object_Detection/ssd300_vgg16_trained_weights_lr01_stratsmall.pth"
size = resize_size
num_classes = 7 # classification labels 1-6 + background class label 0

# Load pretrained model
model = torchvision.models.detection.ssd300_vgg16(weights=SSD300_VGG16_Weights.COCO_V1)

# Define classification head
in_channels = _utils.retrieve_out_channels(model.backbone, (size, size))
num_anchors = model.anchor_generator.num_anchors_per_location()
model.head.classification_head = SSDClassificationHead(in_channels=in_channels,
                                                       num_anchors=num_anchors,
                                                       num_classes=num_classes)

# Transform for images
model.transform.min_size = (size,)
model.transform.max_size = size

# Move model to device
model = model.to(device)

# Load the saved weights
model.load_state_dict(torch.load(model_path))

Downloading: "https://download.pytorch.org/models/ssd300_vgg16_coco-b556d3b4.pth" to /root/.cache/torch/hub/checkpoints/ssd300_vgg16_coco-b556d3b4.pth


100%|██████████| 136M/136M [00:00<00:00, 202MB/s]


<All keys matched successfully>

In [8]:
# Test the model and view output
model.eval()  # Set the model to evaluation mode
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=my_collate_fn)

plt.ion()

label_map2 = {1: "Pedestrian",2: "Car",3: "Biker",4: "Bus",5: "Cart",6: "Skater"}

# Visualize predictions on the test set
with torch.no_grad():
    for images, targets in test_loader:
        images = [image.to(device) for image in images]

        # Get predictions
        predictions = model(images)

        # Extract boxes, scores, and labels
        for img_idx, pred in enumerate(predictions):
            # print(pred)
            if len(pred['boxes']) == 0:  # Check for empty predictions
                continue

            boxes = pred['boxes'].cpu().numpy()
            scores = pred['scores'].cpu().numpy()
            labels = pred['labels'].cpu().numpy()

            # Filter out low-confidence predictions
            threshold = 0.5
            high_conf_idx = np.where(scores > threshold)[0]
            if len(high_conf_idx) == 0:
                print("no predictions above threshold")
                continue  # No predictions above threshold

            boxes = boxes[high_conf_idx]
            scores = scores[high_conf_idx]
            labels = labels[high_conf_idx]

            # Plot the image with bounding boxes
            img = images[img_idx].cpu().numpy().transpose(1, 2, 0)  # Convert back to HWC format
            img = (img * 255).astype(np.uint8)

            # print(f"Box coordinates: {boxes}")
            # print(f"Image dimensions: {img.shape}")

            img_height, img_width, _ = img.shape
            # print(img_height)

            plt.figure(figsize=(10, 10))
            plt.imshow(img)

            for box, score, label in zip(boxes, scores, labels):
                xmin, ymin, xmax, ymax = box

                xmin = np.clip(xmin, 0, img_width)
                xmax = np.clip(xmax, 0, img_width)
                ymin = np.clip(ymin, 0, img_height)
                ymax = np.clip(ymax, 0, img_height)

                label_name = label_map2.get(label, "Unknown")

                plt.gca().add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                                  linewidth=2, edgecolor='red', facecolor='none'))
                plt.text(xmin, ymin, f'{label_name}: {score:.2f}', color='red', fontsize=12,
                        bbox=dict(facecolor='white', alpha=0.7))

            plt.axis('off')
            plt.show(block=True)

# Ensure the video capture object is properly released
if 'cap1' in locals() and cap1.isOpened():
    cap1.release()

if 'cap2' in locals() and cap2.isOpened():
    cap2.release()

if 'cap3' in locals() and cap3.isOpened():
    cap3.release()

if 'cap4' in locals() and cap4.isOpened():
    cap4.release()

if 'cap5' in locals() and cap5.isOpened():
    cap5.release()

if 'cap6' in locals() and cap6.isOpened():
    cap6.release()

if 'cap7' in locals() and cap7.isOpened():
    cap7.release()

if 'cap8' in locals() and cap8.isOpened():
    cap8.release()

if 'cap10' in locals() and cap10.isOpened():
    cap10.release()

Output hidden; open in https://colab.research.google.com to view.

In [9]:

# Test the model and store predictions -- 28m
model.eval()
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=my_collate_fn)

predictions = []
targets = []

# Collect predictions and targets for mAP computation
with torch.no_grad():
    for images, batch_targets in test_loader:
        images = [image.to(device) for image in images]

        # Get predictions
        preds = model(images)
        predictions.extend(preds)
        targets.extend(batch_targets)

In [10]:

## 27m

def compute_iou(box, gt_boxes):
    # Compute IoU between the predicted and ground truth boxes

    xmin = np.maximum(box[0], gt_boxes[:, 0])
    ymin = np.maximum(box[1], gt_boxes[:, 1])
    xmax = np.minimum(box[2], gt_boxes[:, 2])
    ymax = np.minimum(box[3], gt_boxes[:, 3])

    intersection = np.clip(xmax - xmin, 0, None) * np.clip(ymax - ymin, 0, None)
    area_box = (box[2] - box[0]) * (box[3] - box[1])
    area_gt_boxes = (gt_boxes[:, 2] - gt_boxes[:, 0]) * (gt_boxes[:, 3] - gt_boxes[:, 1])

    union = area_box + area_gt_boxes - intersection
    return intersection / (union + 1e-6)

def calculate_map(predictions, targets, num_classes, iou_threshold=0.5):
    true_positives = []
    false_positives = []
    tp_scores = []
    fp_scores = []
    total_ground_truths = []

    for c in range(1, num_classes + 1):
        class_predictions = []
        class_ground_truths = []

        for pred, target in zip(predictions, targets):
            pred_boxes = pred['boxes'][pred['labels'] == c].cpu().numpy()
            pred_scores = pred['scores'][pred['labels'] == c].cpu().numpy()
            gt_boxes = target['boxes'][target['labels'] == c].cpu().numpy()

            total_ground_truths.append(len(gt_boxes))
            if len(pred_boxes) > 0:
                class_predictions.append((pred_boxes, pred_scores))
            if len(gt_boxes) > 0:
                class_ground_truths.append(gt_boxes)

        if len(class_predictions) > 0:
            for pred_boxes, pred_scores in class_predictions:
                for i, box in enumerate(pred_boxes):
                    if class_ground_truths:
                        gt_boxes_flat = np.concatenate(class_ground_truths, axis=0)
                        ious = compute_iou(box, gt_boxes_flat)

                        max_iou = np.max(ious) if len(ious) > 0 else 0
                        best_gt_idx = np.argmax(ious) if len(ious) > 0 else -1

                        if max_iou >= iou_threshold:
                            true_positives.append(1)
                            tp_scores.append(pred_scores[i])

                            # Remove the matched ground truth
                            if best_gt_idx < len(class_ground_truths):
                                del class_ground_truths[best_gt_idx]
                        else:
                            false_positives.append(1)
                            fp_scores.append(pred_scores[i])
                    else:
                        false_positives.append(1)
                        fp_scores.append(pred_scores[i])

        # Handle cases where there are no predictions:
        if len(class_ground_truths) > 0:
            for pred_boxes, pred_scores in class_predictions:
                for i in range(len(pred_boxes)):
                    if i >= len(tp_scores):  # If no true positives were counted
                        false_positives.append(1)
                        fp_scores.append(pred_scores[i])

    # Calculate precision and recall
    tp_count = np.sum(true_positives)
    fp_count = np.sum(false_positives)
    precision = tp_count / (tp_count + fp_count + 1e-6)
    recall = tp_count / (np.sum(total_ground_truths) + 1e-6)

    y_true = [1] * len(true_positives) + [0] * len(false_positives)
    y_score = tp_scores + fp_scores

    # Calculate mean Average Precision
    return precision, recall, average_precision_score(y_true, y_score)

# Calculate mean Average Precision
precision, recall, mAP = calculate_map(predictions, targets, num_classes)
print(f'mAP: {mAP:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}')

mAP: 0.3021, Precision: 0.2942, Recall: 0.9139


In [22]:
def dataset_summary(dataset):
    data_loader = DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=my_collate_fn)
    num_incidents = 0
    num_type = [0, 0, 0, 0, 0, 0]

    for images, targets in data_loader:
        targets = [{k: v.to(device) for k, v in target.items()} for target in targets]

        for target in targets:
          num_incidents += len(target['labels'])
          num_type[0] += target['labels'].tolist().count(1)
          num_type[1] += target['labels'].tolist().count(2)
          num_type[2] += target['labels'].tolist().count(3)
          num_type[3] += target['labels'].tolist().count(4)
          num_type[4] += target['labels'].tolist().count(5)
          num_type[5] += target['labels'].tolist().count(6)


    print(num_incidents, num_type)


# label_map2 = {1: "Pedestrian",2: "Car",3: "Biker",4: "Bus",5: "Cart",6: "Skater"}
dataset_summary(train_dataset)
dataset_summary(valid_dataset)
# dataset_summary(test_dataset)

28980 [19614, 762, 8291, 0, 254, 59]


KeyboardInterrupt: 